# 01 · Filter the raw datasets and build the benchmark inputs (CPU)

Run top to bottom. Set the paths in the first cell (where CrAM's `nq_1000_bge.json` / `trivia_1000_bge.json`
live on this machine). Everything is written under `DATA` by the library functions; you only look at tables.

Steps: import + automatic filter >> inspect >> your optional manual filter and cap >> build pairs and dev/test split
>> human cells from the retrieved chunks >> generation plan >> prompt preview. Then open `02_generate_and_seal.ipynb`
on the GPU box.

In [ ]:
from pathlib import Path
import sys, collections
import pandas as pd
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))

# ---- parameters -------------------------------------------------------------
CRAM_NQ = REPO.parent / "CrAM" / "nq_1000_bge.json"          # raw CrAM release files; change if they sit elsewhere
CRAM_TRIVIA = REPO.parent / "CrAM" / "trivia_1000_bge.json"
MAX_PER_SOURCE = 75                                          # 75 + 75 = 150 pairs
N_DEV = 50                                                   # dev pairs (calibration, heads); the rest is test
# ------------------------------------------------------------------------------

from witness_rag.io import load_config, read_json, read_jsonl, write_jsonl, write_json, data_path
cfg = load_config()
DATA = data_path(cfg)
(DATA / "claims").mkdir(parents=True, exist_ok=True)
print("repo:", REPO); print("data:", DATA)

## 1. Import and automatic filter
Rejection reasons per source. `no_answer_bearing_chunk` = no top-10 chunk states the reference; `wrong_answer_in_real_chunks` = the wrong answer is a partial truth.

In [ ]:
from witness_rag.claims.nq_import import select_pairs

imported, chunks, reasons = [], [], {}
for ds, path in (("nq", CRAM_NQ), ("trivia", CRAM_TRIVIA)):
    p, c, r = select_pairs(read_json(path), ds, None)        # no cap yet: filter first, cap in step 3
    imported += p; chunks += c
    reasons[ds] = {"accepted": len(p), **r}
pd.DataFrame(reasons).fillna(0).astype(int)

## 2. Inspect the candidates

In [ ]:
df = pd.DataFrame(imported)[["source", "topic", "question", "answer_true", "answer_true_aliases", "answer_false", "edit_type"]]
print(len(df), "candidates")
df.sample(15, random_state=0)

## 3. Your filter (optional) and the cap
Drop sources you do not like, or filter `df` any way you want; the cap keeps the first `MAX_PER_SOURCE` per source in id order.

In [ ]:
REJECT = set()                                   # e.g. {"nq:470", "trivia:9646"}
keep = df[~df.source.isin(REJECT)]
keep = keep.groupby("topic", group_keys=False).head(MAX_PER_SOURCE)
kept_sources = set(keep.source)
pairs_imported = [p for p in imported if p["source"] in kept_sources]
chunks_kept = [c for c in chunks if c["source"] in kept_sources]
write_jsonl(DATA / "claims" / "pairs_imported.jsonl", pairs_imported)
write_jsonl(DATA / "claims" / "imported_chunks.jsonl", chunks_kept)
print(len(pairs_imported), "pairs kept")
keep.groupby(["topic", "edit_type"]).size().unstack(fill_value=0)

## 4. Build the canonical pair files and the dev/test split

In [ ]:
from witness_rag.claims.build_claims import build

pairs, splits, stats, errors = build(pairs_imported, N_DEV, cfg["project"]["seed"])
assert not errors, errors[:10]
write_jsonl(DATA / "claims" / "pairs.jsonl", (p.to_dict() for p in pairs))
write_json(DATA / "claims" / "splits.json", splits)
{k: v for k, v in stats.items() if k != "by_topic"} | {"n_dev": splits["n_dev"], "n_test": splits["n_test"]}

## 5. Human cells from the retrieved chunks
H_T = answer-bearing chunk, H_F = same window with the value substituted, H_N = neutral chunks of the same query, G_F = CrAM's fakes. No network.

In [ ]:
from witness_rag.generation.human_cells import build_human_docs

summary = build_human_docs(pairs, cfg, DATA / "human", {c["source"]: c for c in chunks_kept})
flags = read_jsonl(DATA / "human" / "human_flags.jsonl")
print(summary)
pd.DataFrame(flags).groupby("flag").size() if flags else "no flags"

In [ ]:
log = read_jsonl(DATA / "human" / "human_log.jsonl")
docs = {d["doc_id"]: d["text"] for d in read_jsonl(DATA / "human" / "human_docs.jsonl")}
pair = pairs[0]
print("Q:", pair.question, "| true:", pair.answer_true, "| false:", pair.answer_false)
for r in log:
    if r["pair_id"] == pair.pair_id:
        print(f"\n[{r['cell']}] {docs[r['doc_id']][:300]}")

## 6. Generation plan
Who writes what: 3 generators × (3 true + 3 false) independent passages and one coordinated cluster of 5 per pair; shapes balanced across truth.

In [ ]:
from witness_rag.generation.plan import build_plan
from witness_rag.generation.prompts import PromptBook

book = PromptBook()
plan = build_plan(pairs, cfg, book)
write_jsonl(DATA / "production_log_plan.jsonl", plan)
pdf = pd.DataFrame(plan)
print(len(plan), "generations planned")
display(pdf.groupby(["cell", "dependence_level"]).size().rename("n"))
pdf[pdf.dependence_level == "D0"].groupby(["model", "stance", "shape"]).size().unstack(fill_value=0)

In [ ]:
row = plan[0]; pair = next(p for p in pairs if p.pair_id == row["pair_id"])
print("SYSTEM:", book.system, "\n")
other = pair.answer_for("false" if row["stance"] == "true" else "true")
print("USER  :", book.render(row["genre"], pair.question, pair.answer_for(row["stance"]), other, row["length_target"]))

Next: `02_generate_and_seal.ipynb` on a GPU machine with `requirements-gpu.txt`, or `scripts/2_generate_corpus.sh`. Files written here: `data/claims/*`, `data/human/*`, `data/production_log_plan.jsonl`.